In [1]:
# --------------------------------------------------
# 0. Import
# --------------------------------------------------

import pandas as pd

In [2]:
# --------------------------------------------------
# 1. Load input files
# --------------------------------------------------

ml_path = "ml_summary_match_nonmatch.csv"
cnn_path = "cnn_summary_match_nonmatch.csv"
bert_path = "bert_scibert_han_summary_match_nonmatch.csv"

ml_df = pd.read_csv(ml_path, encoding="utf-8-sig")
cnn_df = pd.read_csv(cnn_path, encoding="utf-8-sig")
bert_df = pd.read_csv(bert_path, encoding="utf-8-sig")

print("ML columns:", ml_df.columns.tolist())
print("CNN columns:", cnn_df.columns.tolist())
print("BERT/SciBERT columns:", bert_df.columns.tolist())

ML columns: ['Model', 'Accuracy', 'ROC_AUC', 'Match_Precision', 'Match_Recall', 'Match_F1', 'Match_Support', 'NonMatch_Precision', 'NonMatch_Recall', 'NonMatch_F1', 'NonMatch_Support', 'Overall_PositiveClass_Precision', 'Overall_PositiveClass_Recall', 'Overall_PositiveClass_F1']
CNN columns: ['Model', 'Class', 'Precision', 'Recall', 'F1', 'Support', 'Accuracy', 'ROC_AUC']
BERT/SciBERT columns: ['Dataset', 'Model', 'Class', 'Precision', 'Recall', 'F1', 'Support', 'Accuracy', 'ROC_AUC']


In [3]:
# --------------------------------------------------
# 2. Convert long format to wide format
# --------------------------------------------------

def long_to_wide_summary(df):
    # Remove Dataset column if it exists
    if "Dataset" in df.columns:
        df = df.drop(columns=["Dataset"])
    
    # Pivot Class values into columns
    wide_df = df.pivot(
        index="Model",
        columns="Class",
        values=["Precision", "Recall", "F1", "Support"]
    )
    
    # Flatten multi-level columns
    wide_df.columns = [
        f"{class_name}_{metric}"
        for metric, class_name in wide_df.columns
    ]
    
    wide_df = wide_df.reset_index()
    
    # Add Accuracy and ROC_AUC, one value per model
    extra_cols = []
    for col in ["Accuracy", "ROC_AUC"]:
        if col in df.columns:
            extra = df.groupby("Model")[col].first().reset_index()
            extra_cols.append(extra)
    
    for extra in extra_cols:
        wide_df = wide_df.merge(extra, on="Model", how="left")
    
    # Reorder columns to match ML-style format
    desired_cols = [
        "Model",
        "Accuracy",
        "ROC_AUC",
        "Match_Precision",
        "Match_Recall",
        "Match_F1",
        "Match_Support",
        "NonMatch_Precision",
        "NonMatch_Recall",
        "NonMatch_F1",
        "NonMatch_Support"
    ]
    
    existing_cols = [col for col in desired_cols if col in wide_df.columns]
    
    return wide_df[existing_cols]


cnn_wide = long_to_wide_summary(cnn_df)
bert_wide = long_to_wide_summary(bert_df)

print(cnn_wide)
print(bert_wide)

  Model  Accuracy  ROC_AUC  Match_Precision  Match_Recall  Match_F1  \
0   CNN  0.919535  0.95527         0.800697      0.821451  0.810941   

   Match_Support  NonMatch_Precision  NonMatch_Recall  NonMatch_F1  \
0        19020.0            0.952185         0.945621     0.948891   

   NonMatch_Support  
0           71516.0  
     Model  Accuracy   ROC_AUC  Match_Precision  Match_Recall  Match_F1  \
0     BERT  0.971183  0.990735         0.947337      0.913617  0.930172   
1  SciBERT  0.967273  0.988448         0.935550      0.906677  0.920888   

   Match_Support  NonMatch_Precision  NonMatch_Recall  NonMatch_F1  \
0        19020.0            0.977242         0.986493     0.981845   
1        19020.0            0.975382         0.983388     0.979369   

   NonMatch_Support  
0           71516.0  
1           71516.0  


In [4]:
# --------------------------------------------------
# 3. Keep only the comparable columns from ML results
# --------------------------------------------------

common_cols = [
    "Model",
    "Accuracy",
    "ROC_AUC",
    "Match_Precision",
    "Match_Recall",
    "Match_F1",
    "Match_Support",
    "NonMatch_Precision",
    "NonMatch_Recall",
    "NonMatch_F1",
    "NonMatch_Support"
]

ml_wide = ml_df[common_cols].copy()

print(ml_wide)

                Model  Accuracy   ROC_AUC  Match_Precision  Match_Recall  \
0  LogisticRegression  0.866208  0.918190         0.642665      0.817928   
1        RandomForest  0.866981  0.930802         0.639613      0.840273   

   Match_F1  Match_Support  NonMatch_Precision  NonMatch_Recall  NonMatch_F1  \
0  0.719782          19020            0.947791         0.879048     0.912126   
1  0.726339          19020            0.953653         0.874084     0.912137   

   NonMatch_Support  
0             71516  
1             71516  


In [5]:
# --------------------------------------------------
# 4. Combine ML, CNN, BERT, and SciBERT summaries
# --------------------------------------------------

combined_wide = pd.concat(
    [ml_wide, cnn_wide, bert_wide],
    ignore_index=True
)

# Optional: set model order
model_order = [
    "LogisticRegression",
    "RandomForest",
    "CNN",
    "BERT",
    "SciBERT"
]

combined_wide["Model"] = pd.Categorical(
    combined_wide["Model"],
    categories=model_order,
    ordered=True
)

combined_wide = combined_wide.sort_values("Model").reset_index(drop=True)

combined_wide.to_csv(
    "han_combined_summary_wide.csv",
    index=False,
    encoding="utf-8-sig"
)

print(combined_wide)

                Model  Accuracy   ROC_AUC  Match_Precision  Match_Recall  \
0  LogisticRegression  0.866208  0.918190         0.642665      0.817928   
1        RandomForest  0.866981  0.930802         0.639613      0.840273   
2                 CNN  0.919535  0.955270         0.800697      0.821451   
3                BERT  0.971183  0.990735         0.947337      0.913617   
4             SciBERT  0.967273  0.988448         0.935550      0.906677   

   Match_F1  Match_Support  NonMatch_Precision  NonMatch_Recall  NonMatch_F1  \
0  0.719782        19020.0            0.947791         0.879048     0.912126   
1  0.726339        19020.0            0.953653         0.874084     0.912137   
2  0.810941        19020.0            0.952185         0.945621     0.948891   
3  0.930172        19020.0            0.977242         0.986493     0.981845   
4  0.920888        19020.0            0.975382         0.983388     0.979369   

   NonMatch_Support  
0           71516.0  
1           71516.

In [6]:
# --------------------------------------------------
# 5. Build publication-ready tables
# --------------------------------------------------

def format_value(value, digits=4):
    return f"{float(value):.{digits}f}"


def build_result_table(df, class_prefix):
    rows = []
    
    for _, row in df.iterrows():
        rows.append({
            "Model": row["Model"],
            "Precision": format_value(row[f"{class_prefix}_Precision"]),
            "Recall": format_value(row[f"{class_prefix}_Recall"]),
            "F1": format_value(row[f"{class_prefix}_F1"])
        })
    
    return pd.DataFrame(rows)


match_table = build_result_table(combined_wide, "Match")
nonmatch_table = build_result_table(combined_wide, "NonMatch")

match_table.to_csv(
    "han_positive_match_table.csv",
    index=False,
    encoding="utf-8-sig"
)

nonmatch_table.to_csv(
    "han_negative_nonmatch_table.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Positive Pair (Match) Results")
display(match_table)

print("Negative Pair (NonMatch) Results")
display(nonmatch_table)

Positive Pair (Match) Results


,Model,Precision,Recall,F1
0,LogisticRegression,0.6427,0.8179,0.7198
1,RandomForest,0.6396,0.8403,0.7263
2,CNN,0.8007,0.8215,0.8109
3,BERT,0.9473,0.9136,0.9302
4,SciBERT,0.9356,0.9067,0.9209


Negative Pair (NonMatch) Results


,Model,Precision,Recall,F1
0,LogisticRegression,0.9478,0.8790,0.9121
1,RandomForest,0.9537,0.8741,0.9121
2,CNN,0.9522,0.9456,0.9489
3,BERT,0.9772,0.9865,0.9818
4,SciBERT,0.9754,0.9834,0.9794
